In [1]:
import torch
import torch.nn.functional as F
from torchvision import transforms as T
import numpy as np

import nrrd
import pandas as pd
import os
import sys

import plotly.express as px

# sys.path.append("/mnt/raid/C1_ML_Analysis/source/autoencoder/src")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/")
sys.path.append("/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/nets")


from dl.nets.layers import TimeDistributed
import dl.nets.us_simulation_jit as us_simulation_jit
from dl.nets.us_simu import VolumeSamplingBlindSweep
import SimpleITK as sitk


In [2]:
mount_point = "/mnt/raid/C1_ML_Analysis/"
diffusor_np, diffusor_head = nrrd.read(os.path.join(mount_point, "simulated_data_export/placenta/FAM-025-0351-3_label11.nrrd"))

diffusor_size = diffusor_head['sizes']
diffusor_spacing = np.diag(diffusor_head['space directions'])

diffusor_origin = np.flip(diffusor_head['space origin'], axis=0)
diffusor_end = diffusor_origin + diffusor_spacing * diffusor_size

In [3]:
diffusor_t = torch.tensor(diffusor_np.astype(float)).unsqueeze(0).unsqueeze(0).cuda()
diffusor_origin = torch.tensor(diffusor_origin.copy()).unsqueeze(0).cuda()
diffusor_end = torch.tensor(diffusor_end.copy()).unsqueeze(0).cuda()


In [4]:

# us_simulator = us_simulation_jit.MergedLinearLabel11().eval().cuda()
# us_simulator = us_simulation_jit.MergedLinearLabel11WOG().eval().cuda()
# us_simulator = us_simulation_jit.MergedLinearLabel11PassThrough().eval().cuda()
us_simulator = us_simulation_jit.MergedGuidedLabel11().eval().cuda()

grid, inverse_grid, mask_fan = us_simulator.init_grids(256, 256, 128.0, -30.0, 20.0, 215.0, 0.7853981633974483)
grid = grid.cuda()
inverse_grid = inverse_grid.cuda()
mask_fan = mask_fan.cuda()

us_simulator_td = TimeDistributed(us_simulator, time_dim=2).eval()

vs = VolumeSamplingBlindSweep(mount_point=mount_point).cuda()

/mnt/raid/home/jprieto/envs/torch_us/lib/python3.10/site-packages/lightning/fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
To copy construct from a tensor, it

In [5]:
vs.tags

array(['M', 'L0', 'L1', 'R0', 'R1', 'C1', 'C2', 'C3', 'C4'], dtype=object)

In [6]:
tag_n = 0
sweep, sweep_label = vs.get_sweep(diffusor_t, diffusor_origin, diffusor_end, vs.tags[tag_n], use_random=True, simulator=us_simulator_td, grid=grid, inverse_grid=inverse_grid, mask_fan=mask_fan, return_masked=True)

In [7]:
fig = px.imshow(sweep.squeeze().detach().cpu().numpy(), animation_frame=0, binary_string=True)
fig.show()

In [8]:
fig = px.imshow(sweep_label.squeeze().detach().cpu().numpy(), animation_frame=0, binary_string=True)
fig.show()

In [9]:
# img = sitk.GetImageFromArray(sweep.squeeze().detach().cpu().numpy())
# img.SetSpacing([0.75, 0.75, 1.0])
# print(img)
# sitk.WriteImage(img, "/mnt/raid/C1_ML_Analysis/simulated_data_export/placenta_simu/FAM-025-0351-3_label11/" + vs.tags[tag_n] + ".nrrd")